# Stage 4 Lab — Spot the Bug
### Feature Engineering + Linear Algebra (Topics 5 & 6)

Each section has a **broken cell**. Your job in the breakout room:
1. **Run** it and read the error (or the wrong number).
2. **Find** the one line that's wrong.
3. **Fix** it and re-run.
4. **Explain** to your room *why* it was wrong.

There are **4 bugs**. Work top to bottom. Setup cell first 👇

In [ ]:
# SETUP — run this first, don't change it
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import make_pipeline
print("setup ok — numpy", np.__version__)

---
## BUG 1 — Matrix polynomial (Topic 6, Exercise "Matrix polynomials")

We want to compute the matrix polynomial **p(A) = A² + 2A + I**.
Here A² means *matrix* A times A. Run the cell — the answer is wrong.
**Hint:** how do you multiply two matrices in NumPy?

In [ ]:
A = np.array([[2, 1],
              [0, 3]])
I = np.eye(2, dtype=int)

# BROKEN: this does not compute a real matrix square
pA = A * A + 2 * A + I

print("p(A) =\n", pA)
print("Expected top-right entry is 7, not 1.")

## BUG 2 — Least squares / linear regression (Topics 5 & 6)

Data follows the line **y = 2x + 1**. We fit a regression and predict at x = 1
(true answer = 3). The prediction is off.
**Hint:** the line has a "+1". What lets a model learn that constant?

In [ ]:
X = np.array([[1.], [2.], [3.], [4.]])
y = np.array([3., 5., 7., 9.])          # y = 2x + 1

# BROKEN: something is switched off that the data needs
model = LinearRegression(fit_intercept=False).fit(X, y)

pred = model.predict([[1.0]])[0]
print("prediction at x=1:", round(pred, 3), " (should be 3.0)")

## BUG 3 — One-hot encoding, unseen category (Topic 5, "Categorical encoding")

We fit a one-hot encoder on training cities, then transform a **new** city
that wasn't in training. In real life new categories always appear.
Run it — it crashes.
**Hint:** OneHotEncoder has a parameter for how to handle unknown categories.

In [ ]:
train = pd.DataFrame({"city": ["Fergana", "Tashkent", "Namangan"]})
new   = pd.DataFrame({"city": ["Andijan"]})   # never seen in training

# BROKEN: crashes when it meets a city it wasn't trained on
enc = OneHotEncoder(handle_unknown="error")
enc.fit(train)

print(enc.transform(new).toarray())

## BUG 4 — Data leakage (Topic 5, the concept from the leakage board)

The rule: **split first, fit the scaler on training only.** This code scales
the WHOLE dataset before splitting, so test statistics leak into training.
It runs without error — that's what makes leakage dangerous.
**Hint:** the fix is to put the scaler in a pipeline so it fits inside each
cross-validation fold, never on data it shouldn't see.

In [ ]:
rng = np.random.RandomState(0)
X = rng.rand(200, 5)
y = (X[:, 0] > 0.5).astype(int)

# BROKEN (leaky): the scaler sees ALL the data, including test folds
X_scaled = StandardScaler().fit_transform(X)
leaky_scores = cross_val_score(LogisticRegression(), X_scaled, y, cv=5)
print("leaky CV accuracy:", round(leaky_scores.mean(), 3))

# TODO: replace the two lines above with a pipeline so the scaler
# fits only on the training part of each fold. Compare the score.

---
### When all 4 are fixed
Post in the chat, per bug, **one sentence**: what was wrong and why it matters.
Then compare with the answer key your facilitator shares.